# 04 — Train per-tool world-models (predictors)

Trains three Qwen3-0.6B + LoRA predictors on the cached `(args → output)` pairs from
Notebook 02. After this you have one predictor per tool plus a fitted uncertainty head.
Calibration / threshold selection happens in Notebook 05.

**Time**: ~2-4 hrs each on 1× A100. Run sequentially or in parallel on 2 GPUs.


In [ ]:
import sys, os; sys.path.insert(0, str(os.path.abspath(os.path.join(os.getcwd(), '..'))))
import torch, torch.nn.functional as F
from torch.utils.data import DataLoader
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from dyna_grpo.config import MODEL, PATHS
from dyna_grpo.utils import read_jsonl, set_seed, logger
from dyna_grpo.predictor import ToolPairDataset, collate, ToolPredictor, fidelity_score
set_seed(0)

In [ ]:
EPOCHS = 2
LR = 2e-5
BATCH = 8

def train_one(tool: str):
    print(f'=== Training predictor for {tool} ===')
    train_rows = read_jsonl(Path(PATHS['traces']) / f'{tool}_train.jsonl')
    val_rows   = read_jsonl(Path(PATHS['traces']) / f'{tool}_val.jsonl')
    print(f'  train={len(train_rows)} val={len(val_rows)}')

    tok = AutoTokenizer.from_pretrained(MODEL.predictor_base, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(
        MODEL.predictor_base, torch_dtype=torch.bfloat16, device_map='cuda:0',
        trust_remote_code=True)
    cfg = LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, bias='none',
                     target_modules=list(MODEL.lora_target_modules), task_type='CAUSAL_LM')
    base = get_peft_model(base, cfg)
    h = base.config.hidden_size
    pred = ToolPredictor(base, h).to('cuda:0').to(torch.bfloat16)

    ds_tr = ToolPairDataset(train_rows, tok)
    ds_va = ToolPairDataset(val_rows, tok)
    pad_id = tok.pad_token_id or tok.eos_token_id
    dl_tr = DataLoader(ds_tr, batch_size=BATCH, shuffle=True,
                       collate_fn=lambda b: collate(b, pad_id))
    dl_va = DataLoader(ds_va, batch_size=BATCH, shuffle=False,
                       collate_fn=lambda b: collate(b, pad_id))

    opt = torch.optim.AdamW([p for p in pred.parameters() if p.requires_grad], lr=LR)
    pred.train()
    for epoch in range(EPOCHS):
        for step, batch in enumerate(dl_tr):
            batch = {k: v.to('cuda:0') for k, v in batch.items()}
            out, unc_logit = pred(batch['input_ids'], batch['attention_mask'], batch['labels'])
            loss = out.loss
            loss.backward(); opt.step(); opt.zero_grad()
            if step % 50 == 0:
                print(f'  ep{epoch} step{step} loss={loss.item():.4f}')

    # Save checkpoint
    out_dir = Path(PATHS['ckpts']) / f'predictor_{tool}'
    out_dir.mkdir(parents=True, exist_ok=True)
    pred.base.save_pretrained(out_dir)
    torch.save({'unc_head': pred.unc_head.state_dict(),
                'temperature': pred.temperature.detach().cpu()}, out_dir / 'aux.pt')
    tok.save_pretrained(out_dir)
    return out_dir

for tool in ('calc', 'code', 'search'):
    train_one(tool)
    torch.cuda.empty_cache()

In [ ]:
# Quick sanity: each predictor should produce reasonable output for a held-out args
from peft import PeftModel
for tool in ('calc', 'code', 'search'):
    out_dir = Path(PATHS['ckpts']) / f'predictor_{tool}'
    tok = AutoTokenizer.from_pretrained(out_dir, trust_remote_code=True)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(
        MODEL.predictor_base, torch_dtype=torch.bfloat16, device_map='cuda:0', trust_remote_code=True)
    base = PeftModel.from_pretrained(base, out_dir)
    pred = ToolPredictor(base, base.config.hidden_size).to('cuda:0').to(torch.bfloat16)
    aux = torch.load(out_dir / 'aux.pt', map_location='cuda:0')
    pred.unc_head.load_state_dict(aux['unc_head'])
    pred.temperature.data = aux['temperature'].to(pred.temperature.device).to(pred.temperature.dtype)
    val = read_jsonl(Path(PATHS['traces']) / f'{tool}_val.jsonl')[0]
    o = pred.predict(tok, tool, val['args'])
    print(f'{tool}: predicted={o.text[:80]} unc={o.uncertainty:.3f} | gold={(val["output"] or "")[:80]}')
    del base, pred; torch.cuda.empty_cache()